# ML Pipelines — Production-Ready Code

sklearn Pipelines are the difference between Jupyter notebook code and production code. They prevent data leakage, make code reproducible, and enable easy deployment.

**Topics:** sklearn Pipeline, ColumnTransformer, custom transformers, Pipeline + GridSearchCV, saving pipelines

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (cross_val_score, GridSearchCV, RandomizedSearchCV,
                                      train_test_split)
from sklearn.metrics import classification_report
import joblib
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
print('sklearn Pipeline system ready')

## 1. Why Pipelines Prevent Data Leakage

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ❌ WRONG WAY — data leakage!
scaler = StandardScaler()
X_scaled_leaky = scaler.fit_transform(X)  # uses test data info!
X_train_leaky = X_scaled_leaky[:800]
X_test_leaky  = X_scaled_leaky[800:]
clf_leaky = LogisticRegression(max_iter=1000)
clf_leaky.fit(X_train_leaky, y_train)
score_leaky = clf_leaky.score(X_test_leaky, y_test)

# ✓ CORRECT WAY — Pipeline handles it properly
pipeline = Pipeline([
    ('scaler', StandardScaler()),         # fit ONLY on training data within each CV fold
    ('clf', LogisticRegression(max_iter=1000))
])
scores_correct = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')

print('Data Leakage Demonstration:')
print(f'  Leaky approach (scaled on all data):  {score_leaky:.4f}')
print(f'  Correct pipeline (CV, no leakage):   {scores_correct.mean():.4f} ± {scores_correct.std():.4f}')
print('\n  With Pipeline, fit() is called only on training folds.')
print('  The test fold is transformed using training statistics — no leakage!')

## 2. Custom Transformers

In [ ]:
# Custom transformer: inherits from BaseEstimator + TransformerMixin

class LogTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p transformation to selected columns."""
    
    def __init__(self, shift=1.0):
        self.shift = shift
    
    def fit(self, X, y=None):
        return self  # stateless transformer
    
    def transform(self, X):
        return np.log(X + self.shift)

class OutlierClipper(BaseEstimator, TransformerMixin):
    """Clip outliers at percentile boundaries (fitted on training data)."""
    
    def __init__(self, lower_pct=1, upper_pct=99):
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct
    
    def fit(self, X, y=None):
        self.lower_ = np.percentile(X, self.lower_pct, axis=0)
        self.upper_ = np.percentile(X, self.upper_pct, axis=0)
        return self
    
    def transform(self, X):
        return np.clip(X, self.lower_, self.upper_)

class FeatureSelector(BaseEstimator, TransformerMixin):
    """Select specific columns from DataFrame."""
    
    def __init__(self, columns):
        self.columns = columns
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return X[self.columns]

# Test custom transformers
X_test_data = np.array([[1, 1000], [2, 2000], [3, 500], [100, 10]])  # last row has outliers
log_t = LogTransformer()
clip_t = OutlierClipper(lower_pct=5, upper_pct=95).fit(X_test_data)

print('Original data:'); print(X_test_data)
print('Log transformed:'); print(log_t.fit_transform(X_test_data).round(2))
print('Clipped (5th-95th pct):'); print(clip_t.transform(X_test_data).round(2))

## 3. Full Production Pipeline

In [ ]:
# Realistic dataset with mixed types
n = 2000
df_raw = pd.DataFrame({
    'age':        np.random.randint(18, 75, n),
    'income':     np.random.lognormal(10, 0.8, n),
    'credit':     np.random.normal(680, 100, n).clip(300, 850),
    'employment': np.random.choice(['employed','self_employed','unemployed'], n, p=[0.7,0.2,0.1]),
    'education':  np.random.choice(['bachelor','master','phd','high_school'], n, p=[0.4,0.3,0.1,0.2]),
})

# Inject missing values
df_raw.loc[np.random.choice(n, 150), 'income'] = np.nan
df_raw.loc[np.random.choice(n, 80), 'employment'] = np.nan

y_target = np.random.choice([0, 1], n, p=[0.8, 0.2])

# Split FIRST, then build pipeline (no leakage)
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(df_raw, y_target, test_size=0.2, random_state=42)

numeric_cols = ['age', 'income', 'credit']
categorical_cols = ['employment', 'education']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clipper', OutlierClipper(1, 99)),
    ('log', LogTransformer()),  # log transform after clipping
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols),
])

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=100, random_state=42)),
])

# Cross-validate
scores = cross_val_score(full_pipeline, X_tr, y_tr, cv=5, scoring='roc_auc')
print(f'CV AUC: {scores.mean():.4f} ± {scores.std():.4f}')

# Train and evaluate on test set
full_pipeline.fit(X_tr, y_tr)
print('\nTest set report:')
print(classification_report(y_te, full_pipeline.predict(X_te)))

## 4. GridSearchCV with Pipeline

In [ ]:
# Pipeline + GridSearchCV: tune ALL hyperparameters together (no leakage!)
# Use 'step_name__param' notation to access parameters inside pipeline steps

param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [3, 5],
    'classifier__learning_rate': [0.05, 0.1],
}

grid_search = GridSearchCV(
    full_pipeline, param_grid, cv=3,
    scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search.fit(X_tr, y_tr)

print(f'Best params: {grid_search.best_params_}')
print(f'Best CV AUC: {grid_search.best_score_:.4f}')
print(f'Test AUC:    {grid_search.score(X_te, y_te):.4f}')

# Save the entire pipeline (preprocessing + tuned model) as ONE object
# joblib.dump(grid_search.best_estimator_, 'credit_model_v1.pkl')
# loaded_pipeline = joblib.load('credit_model_v1.pkl')
# loaded_pipeline.predict(new_raw_data)  # automatically preprocesses!

print('\n✓ Pipeline best practices:')
print('  1. Always use Pipeline — prevents leakage, enables proper CV')
print('  2. Save entire pipeline (not just model) with joblib.dump()')
print('  3. Use ColumnTransformer for mixed-type data')
print('  4. Use param_grid with step__param notation for GridSearchCV')